In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29 peft trl triton
    !pip install --no-deps cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf datasets huggingface_hub hf_transfer
    !pip install --no-deps unsloth

In [ ]:
import pandas as pd
import re
import random
from string import punctuation
import numpy as np

In [ ]:
url = ''
file_id = url.split('/')[-2]
read_url='https://drive.google.com/uc?id=' + file_id

data_set = pd.read_excel(read_url, index_col=False)

condition = [
    data_set["rotulo_humano"] == "sem_sintoma", # 0
    data_set["rotulo_humano"] == "sintoma" # 1
]

values = [0, 1]

data_set["classification"] = np.select(condition, values)

data_set

In [ ]:
from unsloth import FastLanguageModel
import torch
from transformers import TextStreamer
from unsloth.chat_templates import get_chat_template

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = 8192,
    load_in_4bit = True,
)

In [ ]:
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
    mapping = {"role" : "from", "content" : "value", "user" : "human", "assistant" : "gpt"}, # ShareGPT style
)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

In [ ]:
def get_prompt(sentence):
  return f"""
    Você é um modelo de IA treinado para classificar frases quanto à ansiedade e depressão.

    Regras:
    - Se a frase indicar ansiedade ou depressão, retorne exatamente "1".
    - Se a frase não indicar ansiedade ou depressão, retorne exatamente "0".
    - Não adicione explicações.

    Frase: "{sentence}"
    """

In [ ]:
from tqdm import tqdm
import torch

answers_llm = {
    "text": data_set["Texto"].tolist(),
    "target": data_set["classification"].tolist(),
    "predicted": []
}

sentences = answers_llm["text"]

FastLanguageModel.for_inference(model)
device = "cuda" if torch.cuda.is_available() else "cpu"
# model.to(device) # This line caused the error and is not needed for 8-bit models

batch_size = 16
for i in tqdm(range(0, len(sentences), batch_size)):
    batch = sentences[i:i + batch_size]

    prompts = [get_prompt(sentence) for sentence in batch]
    inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to(device)
    input_token_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=128)

    decoded_batch = tokenizer.batch_decode(outputs[:, input_token_len:], skip_special_tokens=True)

    decoded_batch = [t.strip() for t in decoded_batch]
    answers_llm["predicted"].extend(decoded_batch)

In [ ]:

# from google.colab import files

df = pd.DataFrame(answers_llm)
df.to_csv('/content/drive/MyDrive/Notebooks/Others/Outputs/respostas_llama3.csv', index=False)

# files.download('respostas_llama3.xlsx')


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings("ignore")

y_true = answers_llm["target"]
y_pred = answers_llm["predicted"]

y_true = [int(x) for x in y_true]
y_pred = [int(x.strip()) if x.strip() in ['0', '1'] else 0 for x in y_pred]

acc = accuracy_score(y_true, y_pred)
print(f"Acurácia: {acc:.4f}")

print("\nRelatório de Classificação:")
print(classification_report(y_true, y_pred, target_names=["Sem ansiedade (0)", "Com ansiedade (1)"]))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
import matplotlib.pyplot as plt

def plot_confusion_matrix(y_preds, y_true, labels=None):
  cm = confusion_matrix(y_true, y_preds, normalize="true")
  fig, ax = plt.subplots(figsize=(6, 6))
  disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
  disp.plot(cmap="Blues", values_format=".2f", ax=ax, colorbar=False)
  plt.title("Normalized confusion matrix")
  plt.show()

In [ ]:
plot_confusion_matrix(y_pred, y_true, labels=["Sem ansiedade (0)", "Com ansiedade (1)"])